# AI.CLASSIFY — BigQuery AI Functions

`AI.CLASSIFY` is a managed scalar function that classifies inputs into categories you provide. It supports single-label and multi-label classification with optional category descriptions.

**When to use it:**
- You have a fixed set of categories and want to classify text or images
- You need multi-label classification (items can belong to multiple categories)
- You want BigQuery to auto-optimize classification prompts

**Alternatives:**
- [`AI.GENERATE`](../ai_generate/) — Full control with output_schema for custom classification logic
- [`AI.IF`](../ai_if/) — Boolean classification (yes/no)
- [`AI.SCORE`](../ai_score/) — Numeric scoring instead of categorical

**Featured in:** [Content Analysis Pipeline](../../workflows/content_analysis/) | [Document Intelligence](../../workflows/document_intelligence/) | [Content Moderation](../../workflows/content_moderation/)

**Multimodal:** Supports document, image, and video input via [ObjectRef](../../RESOURCES.md#objectref-and-objectrefruntime-schema-reference). Pass a STRUCT input with ObjectRefRuntime fields to classify documents, images, or video.

**References:** [Full syntax reference](../../RESOURCES.md) | [Official documentation](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-classify) | [Setup guide](../../setup/)

---
## Setup

Set your project and location, authenticate, and create a temporary dataset for this notebook.

> This function doesn't require a connection or model for SQL usage — it uses end-user credentials automatically. The [multimodal examples](#examples--multimodal-with-objectref) and BigFrames section add a connection for GCS and Vertex AI access. See the [Setup Reference](../../setup/) for details.

In [1]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks
CONNECTION_ID = 'bq_ai_functions'  # Shared connection (used by multimodal examples and BigFrames)
BUCKET = PROJECT_ID  # GCS bucket (same name as project)

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [2]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install('google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm', 'bigframes', 'google-cloud-storage')

In [3]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [4]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready


---
## Examples — SQL

Progressive examples from simplest to most advanced. Each cell adds one new concept.

### 1. Simple classification

`AI.CLASSIFY` takes an input and an array of categories. Returns the best-matching category as STRING.

In [5]:
query = """
SELECT
  review,
  AI.CLASSIFY(
    review,
    ['positive', 'negative', 'neutral']
  ) AS sentiment
FROM UNNEST([
  'Amazing product, exceeded expectations!',
  'Terrible quality, fell apart immediately.',
  'It works as expected. Nothing special.'
]) AS review
"""
client.query(query).to_dataframe()

,review,sentiment
0,"Terrible quality, fell apart immediately.",negative
1,It works as expected. Nothing special.,neutral
2,"Amazing product, exceeded expectations!",positive


### 2. Categories with descriptions

Provide descriptions to guide the model. Use `STRUCT` pairs: `(category, description)`.

In [6]:
query = """
SELECT
  ticket,
  AI.CLASSIFY(
    ticket,
    [('billing', 'Issues with charges, invoices, or payments'),
     ('technical', 'Product bugs, errors, or how-to questions'),
     ('shipping', 'Delivery, tracking, or logistics issues'),
     ('other', 'Anything not matching the above categories')]
  ) AS category
FROM UNNEST([
  'I was charged twice for my subscription.',
  'The app crashes when I try to upload files.',
  'My package has been in transit for 3 weeks.',
  'Can I get a gift card for my friend?'
]) AS ticket
"""
client.query(query).to_dataframe()

,ticket,category
0,My package has been in transit for 3 weeks.,shipping
1,I was charged twice for my subscription.,billing
2,Can I get a gift card for my friend?,other
3,The app crashes when I try to upload files.,technical


### 3. Multi-label classification

Set `output_mode => 'multi'` to allow items to belong to multiple categories. Returns `ARRAY<STRING>`.

In [7]:
query = """
SELECT
  article,
  AI.CLASSIFY(
    article,
    ['technology', 'business', 'science', 'politics', 'health'],
    output_mode => 'multi'
  ) AS categories
FROM UNNEST([
  'New AI regulations proposed by the EU could reshape the tech industry.',
  'Scientists develop a cheaper solar panel that could boost clean energy stocks.',
  'Study shows regular exercise reduces risk of cognitive decline.'
]) AS article
"""
client.query(query).to_dataframe()

,article,categories
0,Scientists develop a cheaper solar panel that could boost clean energy stocks.,"[technology, business, science]"
1,New AI regulations proposed by the EU could reshape the tech industry.,"[technology, business, politics]"
2,Study shows regular exercise reduces risk of cognitive decline.,"[science, health]"


### 4. Single-label with array output

`output_mode => 'single'` returns an `ARRAY<STRING>` of length 1 — useful when you need consistent array output type.

In [8]:
query = """
SELECT
  email,
  AI.CLASSIFY(
    email,
    ['spam', 'promotional', 'personal', 'work'],
    output_mode => 'single'
  ) AS category
FROM UNNEST([
  'You have won a million dollars! Click here NOW!',
  'Sale: 50% off all items this weekend only.',
  'Hey, are we still meeting for dinner tonight?',
  'Please review the Q3 budget spreadsheet by Friday.'
]) AS email
"""
client.query(query).to_dataframe()

,email,category
0,You have won a million dollars! Click here NOW!,[spam]
1,Please review the Q3 budget spreadsheet by Friday.,[work]
2,"Hey, are we still meeting for dinner tonight?",[personal]
3,Sale: 50% off all items this weekend only.,[promotional]


### 5. Specifying an endpoint

Override the auto-selected model.

In [9]:
query = """
SELECT
  product,
  AI.CLASSIFY(
    product,
    ['Electronics', 'Clothing', 'Home & Kitchen', 'Sports', 'Books'],
    endpoint => 'gemini-2.5-flash'
  ) AS category
FROM UNNEST([
  'Wireless noise-cancelling headphones',
  'Organic cotton t-shirt',
  'Stainless steel cooking pot set',
  'Yoga mat with carrying strap',
  'Data engineering textbook'
]) AS product
"""
client.query(query).to_dataframe()

,product,category
0,Organic cotton t-shirt,Clothing
1,Data engineering textbook,Books
2,Stainless steel cooking pot set,Home & Kitchen
3,Wireless noise-cancelling headphones,Electronics
4,Yoga mat with carrying strap,Sports


---
## Examples — Multimodal with ObjectRef

`AI.CLASSIFY` can classify documents, images, and video stored in Cloud Storage. Create an **object table** to reference GCS files, then use `EXTERNAL_OBJECT_TRANSFORM` to get signed references that `AI.CLASSIFY` can read.

```
Object table (GCS URIs + connection)
  → EXTERNAL_OBJECT_TRANSFORM(TABLE, ['SIGNED_URL'])
    → ref column (ObjectRef with signed URL)
      → AI.CLASSIFY(ref, categories)
```

See the [ObjectRef reference](../../RESOURCES.md#objectref-and-objectrefruntime-schema-reference) for details.

### Multimodal setup — connection, documents, and object table

ObjectRef requires a [Cloud resource connection](../../setup/) to access GCS. The cells below create a connection, upload sample documents, and create an object table.

In [25]:
import subprocess as _sp

# Create the connection (idempotent — succeeds even if it already exists)
_sp.run(
    ['bq', 'mk', '--connection', '--location', LOCATION,
     '--connection_type', 'CLOUD_RESOURCE',
     '--project_id', PROJECT_ID, CONNECTION_ID],
    capture_output=True, text=True
)

# Get the connection's service account
import json as _json
_conn = _sp.run(
    ['bq', 'show', '--connection', '--format=json',
     f'{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}'],
    capture_output=True, text=True
)
sa = _json.loads(_conn.stdout)['cloudResource']['serviceAccountId']
print(f'Connection SA: {sa}')

# Grant GCS read access (needed for ObjectRef to read documents)
_sp.run(
    ['gcloud', 'projects', 'add-iam-policy-binding', PROJECT_ID,
     f'--member=serviceAccount:{sa}', '--role=roles/storage.objectViewer', '--quiet'],
    capture_output=True, text=True
)
print('Granted roles/storage.objectViewer')

Connection SA: bqcx-1026793852137-hx0g@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Granted roles/storage.objectViewer


In [26]:
from google.cloud import storage as _storage
from pathlib import Path

_gcs = _storage.Client(project=PROJECT_ID)
_bucket = _gcs.bucket(BUCKET)
_prefix = 'bq_ai_functions/ai_classify'

# Find the data directory (works from repo checkout or notebook directory)
_data = Path('../../data/documents')
if not _data.exists():
    _data = Path('data/documents')

# Upload one invoice and one receipt
for subdir, filename in [('invoices', 'invoice_001.pdf'), ('receipts', 'receipt_001.pdf')]:
    blob = _bucket.blob(f'{_prefix}/{filename}')
    if not blob.exists():
        blob.upload_from_filename(str(_data / subdir / filename))
        print(f'Uploaded {filename} → gs://{BUCKET}/{_prefix}/{filename}')
    else:
        print(f'Already exists: gs://{BUCKET}/{_prefix}/{filename}')

Already exists: gs://statmike-mlops-349915/bq_ai_functions/ai_classify/invoice_001.pdf
Already exists: gs://statmike-mlops-349915/bq_ai_functions/ai_classify/receipt_001.pdf


In [27]:
# Create an object table pointing to the uploaded documents
client.query(f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.ai_classify_docs`
WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['gs://{BUCKET}/bq_ai_functions/ai_classify/*.pdf']
)
""").result()
print('Object table ai_classify_docs ready')

Object table ai_classify_docs ready


### 6. Classify a document

Use `EXTERNAL_OBJECT_TRANSFORM` to get a signed `ref` from the object table, then pass it directly to `AI.CLASSIFY`.

In [28]:
query = f"""
SELECT
  uri,
  AI.CLASSIFY(
    docs.ref,
    ['invoice', 'receipt', 'contract', 'report']
  ) AS document_type
FROM
  EXTERNAL_OBJECT_TRANSFORM(TABLE `{PROJECT_ID}.{DATASET_ID}.ai_classify_docs`,
                            ['SIGNED_URL']) AS docs
"""
client.query(query).to_dataframe()

,uri,document_type
0,gs://statmike-mlops-349915/bq_ai_functions/ai_classify/invoice_001.pdf,invoice
1,gs://statmike-mlops-349915/bq_ai_functions/ai_classify/receipt_001.pdf,receipt


### 7. Classify documents with category descriptions

Add descriptions to guide the model when distinguishing between similar document types.

In [29]:
query = f"""
SELECT
  uri,
  AI.CLASSIFY(
    docs.ref,
    [('invoice', 'A bill for goods or services with line items, totals, and payment terms'),
     ('receipt', 'A proof of purchase showing items bought and amount paid'),
     ('contract', 'A legal agreement between parties with terms and conditions'),
     ('report', 'An analytical document with findings, data, or recommendations')]
  ) AS document_type
FROM
  EXTERNAL_OBJECT_TRANSFORM(TABLE `{PROJECT_ID}.{DATASET_ID}.ai_classify_docs`,
                            ['SIGNED_URL']) AS docs
"""
client.query(query).to_dataframe()

,uri,document_type
0,gs://statmike-mlops-349915/bq_ai_functions/ai_classify/invoice_001.pdf,invoice
1,gs://statmike-mlops-349915/bq_ai_functions/ai_classify/receipt_001.pdf,receipt


---
## Examples — `%%bigquery` Magics

The same examples using IPython magic commands. Magics let you write SQL directly in notebook cells without Python string wrapping.

Key patterns:
- `%%bigquery` — run SQL, display results inline
- `%%bigquery df` — run SQL, capture results into a pandas DataFrame

### Single-label classification

In [15]:
%%bigquery --project {PROJECT_ID}

SELECT
  review,
  AI.CLASSIFY(
    review,
    ['positive', 'negative', 'neutral']
  ) AS sentiment
FROM UNNEST([
  'Love it!', 'Terrible.', 'It was okay.', 'Best purchase ever!'
]) AS review

Query is running:   0%|          |

Downloading:   0%|          |

,review,sentiment
0,Love it!,positive
1,Best purchase ever!,positive
2,It was okay.,neutral
3,Terrible.,negative


### Multi-label classification

In [16]:
%%bigquery df_classified --project {PROJECT_ID}

SELECT
  article,
  AI.CLASSIFY(
    article,
    ['technology', 'business', 'science', 'health'],
    output_mode => 'multi'
  ) AS categories
FROM UNNEST([
  'AI regulations could reshape tech industry profits.',
  'New drug shows promise in clinical trials for rare disease.'
]) AS article

Query is running:   0%|          |

Downloading:   0%|          |

In [17]:
df_classified

,article,categories
0,New drug shows promise in clinical trials for rare disease.,"[science, health]"
1,AI regulations could reshape tech industry profits.,"[technology, business]"


---
## Examples — BigFrames

BigFrames wraps `AI.CLASSIFY` via `bbq.ai.classify()`. It returns a Series of STRING directly.

**Key patterns:**
- Takes `categories` as a list/tuple
- Returns STRING directly (single label)
- Example: `bbq.ai.classify(df["text"], ["positive", "negative", "neutral"])`

In [30]:
import bigframes.pandas as bpd
import bigframes.bigquery as bbq

bpd.close_session()  # Reset session to apply connection settings
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION
bpd.options.bigquery.bq_connection = f'{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}'

### Classification

In [31]:
df = bpd.DataFrame({'review': [
    'Amazing product!', 'Terrible quality.', 'Pretty good.', 'Awful experience.'
]})

df['sentiment'] = bbq.ai.classify(df['review'], ['positive', 'negative', 'neutral'])
df.to_pandas()

/usr/local/google/home/statmike/Git/google_projects/projects/bq-ai-functions/.venv/lib/python3.13/site-packages/bigframes/session/__init__.py:384: FutureWarning: You are using the BigFrames session default connection: statmike-
mlops-349915.US.bq_ai_functions,             which can be different
from the BigQuery project default connection.             This default
connection may change in the future.
  warnings.warn(msg, category=FutureWarning)


,review,sentiment
0,Amazing product!,positive
1,Terrible quality.,negative
2,Pretty good.,positive
3,Awful experience.,negative


---
## Cleanup

Drop the object table and remove GCS files uploaded for the multimodal examples.

In [32]:
# Drop the object table
client.query(f"DROP EXTERNAL TABLE IF EXISTS `{PROJECT_ID}.{DATASET_ID}.ai_classify_docs`").result()
print('Dropped ai_classify_docs')

# Delete GCS files uploaded for multimodal examples
from google.cloud import storage

gcs_client = storage.Client(project=PROJECT_ID)
bucket = gcs_client.bucket(BUCKET)
prefix = 'bq_ai_functions/ai_classify/'

blobs = list(bucket.list_blobs(prefix=prefix))
for blob in blobs:
    blob.delete()
    print(f'Deleted gs://{BUCKET}/{blob.name}')
if not blobs:
    print('No GCS files to clean up')


Dropped ai_classify_docs
Deleted gs://statmike-mlops-349915/bq_ai_functions/ai_classify/invoice_001.pdf
Deleted gs://statmike-mlops-349915/bq_ai_functions/ai_classify/receipt_001.pdf


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [ ]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')